#GOAL: implement cropr flow for baseline 2
- design a cropr learned token pruner for the attention modules of the models.  i think we can just call them and then rewrite the attn forward pass or adjust block section?  not sure
- we need to consider if we are doing a full evaluation of all 3 schedules for token cropr like gradual with last layer fusion, gradual without fusion and 3 stage scheduling.  
- adjust report_2_cropr_runner.py so that we include metrics on the tokens pruned and keep all original metrics such as accuracy latency thruouput memory and FLOPS. 
- metrics for token pruning: put the pruning schedule alongside accuracy and throughput 

In [ ]:
import os
import inspect
from pathlib import Path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["XFORMERS_DISABLED"] = "1"

lazy import torch
lazy from torch import nn
lazy from torch.nn import functional as F
lazy from transformers import AutoConfig, AutoModel
lazy from safetensors.torch import load_file

ROOT = Path.cwd().parent
SOURCE = ROOT / "models" / "source"
CHECKPOINTS = ROOT / "models" / "checkpoints"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

load models


In [4]:
class FrozenBackbone(nn.Module):
    def __init__(self, name):
        super().__init__()
        self.name = name

        if name == "dinov2":
            self.model = torch.hub.load(
                str(SOURCE / "dinov2"),
                "dinov2_vitb14",
                source="local",
                pretrained=False,
            )

            weights = torch.load(
                CHECKPOINTS / "dinov2" / "dinov2_vitb14_pretrain.pth",
                map_location="cpu",
                weights_only=True,
            )
            self.model.load_state_dict(weights, strict=True)

            self.feature_dim = self.model.embed_dim
            self.patch_size = 14
            mean = (0.485, 0.456, 0.406)
            std = (0.229, 0.224, 0.225)

        elif name == "siglip2":
            folder = CHECKPOINTS / "siglip2"

            config = AutoConfig.from_pretrained(
                str(folder),
                local_files_only=True,
            )

            full_model = AutoModel.from_config(config)
            weights = load_file(
                str(folder / "model.safetensors"),
                device="cpu",
            )
            full_model.load_state_dict(weights, strict=True)

            self.model = full_model.vision_model
            self.feature_dim = config.vision_config.hidden_size
            self.patch_size = config.vision_config.patch_size

            del full_model, weights

            mean = (0.5, 0.5, 0.5)
            std = (0.5, 0.5, 0.5)

        elif name == "radio":
            checkpoint = (
                CHECKPOINTS / "radio" / "radio-v2.5-b_half.pth.tar"
            ).resolve()

            if not checkpoint.is_file():
                raise FileNotFoundError(checkpoint)

            self.model = torch.hub.load(
                str(SOURCE / "radio"),
                "radio_model",
                source="local",
                version=str(checkpoint),
                adaptor_names=None,
            )

            self.feature_dim = self.model.embed_dim
            self.patch_size = 16

            # RADIO performs its own normalization.
            mean = (0.0, 0.0, 0.0)
            std = (1.0, 1.0, 1.0)

        else:
            raise ValueError(f"Unknown model: {name}")

        self.model.float().requires_grad_(False).eval()

        if name == "radio":
            self.model.input_conditioner.dtype = torch.float32

        self.register_buffer(
            "mean", torch.tensor(mean).view(1, 3, 1, 1)
        )
        self.register_buffer(
            "std", torch.tensor(std).view(1, 3, 1, 1)
        )

    def train(self, mode=True):
        # Preserve Report 1's backbone evaluation behavior.
        super().train(False)
        return self

    # No @torch.no_grad(): LoRA needs autograd.
    def forward(self, images):
        if images.shape[-2:] != (448, 448):
            raise ValueError("Expected 448 × 448 images.")

        images = (images - self.mean) / self.std

        if self.name == "dinov2":
            patches = self.model.forward_features(
                images
            )["x_norm_patchtokens"]

        elif self.name == "siglip2":
            patches = self.model(
                pixel_values=images,
                interpolate_pos_encoding=True,
                return_dict=True,
            ).last_hidden_state

        else:
            _, patches = self.model(images, feature_fmt="NLC")

        grid = (
            images.shape[-2] // self.patch_size,
            images.shape[-1] // self.patch_size,
        )
        return patches.float(), grid

task heads


In [5]:
class ClassificationHead(nn.Module):
    def __init__(self, feature_dim, num_classes=1000):
        super().__init__()
        self.classifier = nn.Linear(feature_dim, num_classes)

    def forward(self, patches):
        return self.classifier(patches.mean(dim=1))


class SegmentationHead(nn.Module):
    def __init__(self, feature_dim, num_classes=150, width=256):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.Conv2d(feature_dim, width, 1, bias=False),
            nn.GroupNorm(32, width),
            nn.GELU(),
            nn.Conv2d(width, width, 3, padding=1, bias=False),
            nn.GroupNorm(32, width),
            nn.GELU(),
            nn.Conv2d(width, num_classes, 1),
        )

    def forward(self, patches, grid_size, output_size):
        batch, count, channels = patches.shape
        height, width = grid_size

        if count != height * width:
            raise ValueError("Tokens do not match the spatial grid.")

        features = patches.transpose(1, 2).reshape(
            batch, channels, height, width
        )

        return F.interpolate(
            self.decoder(features),
            size=output_size,
            mode="bilinear",
            align_corners=False,
        )


class BaselineModel(nn.Module):
    def __init__(self, name):
        super().__init__()
        self.backbone = FrozenBackbone(name)
        dim = self.backbone.feature_dim
        self.classification_head = ClassificationHead(dim)
        self.segmentation_head = SegmentationHead(dim)

    def forward(self, images, task, output_size=None):
        patches, grid = self.backbone(images)

        if task == "classification":
            return self.classification_head(patches)

        if task == "segmentation":
            if output_size is None:
                output_size = images.shape[-2:]

            return self.segmentation_head(
                patches, grid, output_size
            )

        raise ValueError(f"Unknown task: {task}")

In [6]:
# Keep on CPU while inspecting.
models = {
    name: BaselineModel(name)
    for name in ("dinov2", "siglip2", "radio")
}

dino = models["dinov2"]
siglip = models["siglip2"]
radio = models["radio"]

attention_modules = {
    "dinov2": dino.backbone.model.blocks[0].attn,
    "siglip2": siglip.backbone.model.encoder.layers[0].self_attn,
    "radio": radio.backbone.model.model.blocks[0].attn,
}

for name, attention in attention_modules.items():
    print(f"\n--- {name} ---")
    print(type(attention))
    print(attention)
    print("Source:", inspect.getfile(type(attention)))

FileNotFoundError: [Errno 2] No such file or directory: 'd:\\School Stuff\\shanghai work\\Deep Learning Project\\report_2\\models\\source\\dinov2\\hubconf.py'